In [ ]:
import numpy as np
import re
import os
import ot

from gerrychain import Partition, Graph, MarkovChain, updaters, constraints, accept
from gerrychain.proposals import ReCom
from gerrychain.constraints import contiguous
from gerrychain.partition import recursive_tree_part

from functools import partial
import pandas

import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations

from binary_ensemble import BendlEncoder, BendlDecoder
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, dijkstra
from scipy.sparse.linalg import eigs
import altair as alt
import pandas as pd
import random



In [ ]:
def load_grid(filepath):
    with open(filepath,'r') as file:
        lines = file.readlines()
    return lines

def assignment_to_line(assignment, rows, cols):
    return ",".join(str(assignment[r*cols + c]) for r in range(rows) for c in range(cols))

def build_edges(rows, cols):
    edges = []
    for r in range(rows):
        for c in range(cols):
            n = r * cols + c
            if c < cols - 1:
                edges.append((n, n + 1))
            if r < rows - 1:
                edges.append((n, n + cols))
    edges = np.array(edges)
    return edges[:, 0], edges[:, 1]
 
def cut_edges(a, U, V):
    return int(np.sum(a[U] != a[V]))

def ollivier_ricci_edge(P, D, x, y):
    supp = np.union1d(np.flatnonzero(P[x]), np.flatnonzero(P[y]))
    m_x, m_y = P[x, supp], P[y, supp]
    D_sub = D[np.ix_(supp, supp)]
    w1 = ot.emd2(m_x, m_y, D_sub)
    return 1 - w1 / D[x, y]

def spectral_gap(P):
    return 1 - np.sort(np.linalg.eigvals(P))[::-1][1]

def spectra(P):
    return np.sort(np.linalg.eigvals(P))[::-1]

def spectral_gap_sparse(P, k=2):
    # A full dense eigendecomposition (spectral_gap above) needs O(N^2) memory and
    # O(N^3) time, infeasible once N is in the hundreds of thousands. ARPACK (via
    # scipy.sparse.linalg.eigs) finds just the top-|k| eigenvalues of a large sparse
    # matrix directly, which is all spectral_gap actually needs.
    vals = eigs(P.T, k=k, which='LM', return_eigenvectors=False)
    vals_sorted = sorted(vals, key=lambda z: -z.real)
    return 1 - vals_sorted[1]

def ollivier_ricci_edge_sparse(P, x, y, limit):
    # Same Ollivier-Ricci curvature as ollivier_ricci_edge, but for graphs too large to
    # precompute a full all-pairs distance matrix D (which is generically dense even
    # when P itself is sparse -- shortest-path distances between almost every pair of
    # nodes in a connected graph are nonzero). Distances are instead found on demand,
    # bounded to a local search radius (`limit`) around the edge's support.
    supp = np.union1d(P.indices[P.indptr[x]:P.indptr[x+1]], P.indices[P.indptr[y]:P.indptr[y+1]])
    idx_of = {node: i for i, node in enumerate(supp)}
    if x not in idx_of or y not in idx_of:
        return None
    D_local = dijkstra(P, directed=True, indices=supp, limit=limit)
    dxy = D_local[idx_of[x], y]
    if not np.isfinite(dxy) or dxy == 0:
        return None
    D_sub = D_local[:, supp]
    m_x = np.asarray(P[x, supp].todense()).flatten()
    m_y = np.asarray(P[y, supp].todense()).flatten()
    w1 = ot.emd2(m_x, m_y, D_sub)
    val = 1 - w1 / dxy
    return val if np.isfinite(val) else None

def parse(line):
    return np.array([int(x) for x in line.strip().split(',')],dtype=int)


def get_cut_edges(assignments, rows, cols):
    U, V = build_edges(rows, cols)
    return [cut_edges(parse(line), U, V) for line in assignments]

def district_nodes(a,d):
    return np.flatnonzero(a==d)

def induced_edges(U,V,nodes):
    keep = np.isin(U,nodes) & np.isin(V,nodes)
    return U[keep], V[keep]

def laplacian(nodes, U_sub, V_sub):
    index_of = {node: i for i, node in enumerate(nodes)}
    n = len(nodes)
    L = np.zeros((n,n),dtype=int)
    for u, v in zip(U_sub,V_sub):
        i,j = index_of[u],index_of[v]
        L[i,i] += 1
        L[j,j] += 1
        L[i,j] -= 1
        L[j,i] -= 1
    return L 

def spanning_trees(nodes, U_sub, V_sub):
    if len(nodes) <= 1:
        return 1
    L = laplacian(nodes, U_sub, V_sub)
    minor = L[1:, 1:]                 # delete row 0 and column 0
    return round(np.linalg.det(minor))  # Kirchhoff's theorem

def plan_spanning_trees(a, U, V):
    total = 1
    for d in np.unique(a):
        nodes = district_nodes(a, d)
        U_sub, V_sub = induced_edges(U, V, nodes)
        total *= spanning_trees(nodes, U_sub, V_sub)
    return total

def get_spanning_trees(assignments, rows, cols):
    U, V = build_edges(rows, cols)
    return [plan_spanning_trees(parse(line), U, V) for line in assignments]


def district_adjacency(a, U, V):
    pairs = set()
    for u, v in zip(U, V):
        du, dv = a[u], a[v]
        if du != dv:
            pairs.add(tuple(sorted((du, dv))))
    return pairs

def adjacent_district_counts(a, U, V):
    pairs = district_adjacency(a, U, V)
    counts = {d: 0 for d in np.unique(a)}
    for d1, d2 in pairs:
        counts[d1] += 1
        counts[d2] += 1
    return counts

def get_adjacent_district_counts(assignments, rows, cols):
    U, V = build_edges(rows, cols)
    return [list(adjacent_district_counts(parse(line), U, V).values()) for line in assignments]

def ground_truth_distribution(assignments, rows, cols):
    weights = np.array(get_spanning_trees(assignments, rows, cols), dtype=float)
    Z = weights.sum()
    return weights / Z          

def uniform_distribution(assignments):
    n = len(assignments)
    return np.full(n, 1.0 / n)  

def weighted_stat_distribution(stat_values, weights):
    dist = {}
    for val, w in zip(stat_values, weights):
        dist[val] = dist.get(val, 0.0) + w
    return dist   # {stat value: probability mass under this weighting}

def build_plan_index(assignments, rows, cols):
    index = {}
    for i, line in enumerate(assignments):
        key = canonicalize(parse(line))
        index[key] = i
    return index

def canonicalize(a):
    mapping = {}
    next_label = 1
    canon = np.empty_like(a)
    for i, val in enumerate(a):
        if val not in mapping:
            mapping[val] = next_label
            next_label += 1
        canon[i] = mapping[val]
    return tuple(canon)


def lookup_chain_plans(chain_assignments):
    indices = []
    for a in chain_assignments:          # CSV strings, or int arrays/lists from a BEN decode
        arr = parse(a) if isinstance(a, str) else np.asarray(a)
        key = canonicalize(arr)

        if key not in plan_index:
            raise ValueError(f"chain visited a plan not in the enumeration: {key}")
        indices.append(plan_index[key])
    return indices

def plan_ids_from_all_runs(all_runs):
    # adapter for old in-memory string-assignment runs -> same shape as plan_ids_by_seed
    return {seed: np.asarray(lookup_chain_plans(a)) for seed, a in all_runs.items()}

def ground_truth_stat_dist(stat_vals, weights, n_bins):
    dist = np.zeros(n_bins)
    for v, w in zip(stat_vals, weights):
        dist[v] += w
    return dist

def enumerate_spanning_trees(nodes, U_sub, V_sub):
    edge_list = [(U_sub[i], V_sub[i], i) for i in range(len(U_sub))]

    def connected(node_list, edges):
        # Union-find reachability check: lets us bail out of a doomed branch
        # immediately instead of peeling edges off one at a time until none are left.
        if len(node_list) <= 1:
            return True
        parent = {v: v for v in node_list}
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        remaining = len(node_list)
        for u, v, _ in edges:
            ru, rv = find(u), find(v)
            if ru != rv:
                parent[ru] = rv
                remaining -= 1
                if remaining == 1:
                    return True
        return False

    def recurse(node_list, edges):
        if len(node_list) == 1:
            return [[]]
        if not connected(node_list, edges):
            return []
        u, v, eid = edges[0]
        rest = edges[1:]

        trees_without = recurse(node_list, rest)

        merged_nodes = [x for x in node_list if x != v]
        merged_edges = []
        for (a, b, e2) in rest:
            a = u if a == v else a
            b = u if b == v else b
            if a == b:
                continue
            merged_edges.append((a, b, e2))
        trees_with = [
            t + [(U_sub[eid], V_sub[eid], eid)]   # true original endpoints, looked up by eid
            for t in recurse(merged_nodes, merged_edges)
        ]
        return trees_without + trees_with

    return recurse(list(nodes), edge_list)


def balanced_edges(tree, target_size):
    adj = {}
    for (u, v, eid) in tree:
        adj.setdefault(u, []).append((v, eid))
        adj.setdefault(v, []).append((u, eid))

    results = []
    for (u, v, eid) in tree:
        visited = {u}
        stack = [u]
        while stack:
            cur = stack.pop()
            for nbr, e2 in adj[cur]:
                if e2 == eid:          # skip the removed edge
                    continue
                if nbr not in visited:
                    visited.add(nbr)
                    stack.append(nbr)
        if len(visited) == target_size:
            results.append((eid, frozenset(visited)))
    return results

def apply_split(a, W, i, j, side_nodes):
    new_a = a.copy()
    other_side = set(W.tolist()) - side_nodes
    new_a[list(side_nodes)] = i
    new_a[list(other_side)] = j
    return new_a

def shape_split_weights(W, U_sub, V_sub, target_size):
    # Weight of each balanced 2-way split of W, and the total spanning-tree count of
    # W (balanced + unbalanced) needed to normalize the RevReCom proposal - purely a
    # function of W's shape (grid geometry), independent of which plan or district
    # labels it came from. Callers can cache this pair keyed by tuple(W) across many
    # plans that happen to merge the same pair of grid cells into the same shape.
    trees = enumerate_spanning_trees(W, U_sub, V_sub)
    total_trees = len(trees)
    raw_weights = {}
    for tree in trees:
        cuts = balanced_edges(tree, target_size)
        if not cuts:
            continue
        w = 1.0 / len(cuts)
        for _eid, side_nodes in cuts:
            raw_weights[side_nodes] = raw_weights.get(side_nodes, 0.0) + w
    return total_trees, raw_weights

def split_distribution(a, W, U_sub, V_sub, i, j, target_size, shape_cache=None):
    # RevReCom transition weights for the (i, j) proposal: pick a district pair,
    # sample uniformly among ALL spanning trees of their union (not just the ones
    # with a balanced cut - unbalanced trees self-loop), then accept a balanced
    # split with probability 1/s where s is the number of grid edges between the
    # two proposed districts (rejections also self-loop). The returned dict's
    # values therefore do NOT sum to 1 - the shortfall from 1 is this pair's
    # self-loop probability mass, added to the diagonal by the caller.
    key = tuple(int(x) for x in W)
    cached = None if shape_cache is None else shape_cache.get(key)
    if cached is None:
        cached = shape_split_weights(W, U_sub, V_sub, target_size)
        if shape_cache is not None:
            shape_cache[key] = cached
    total_trees, raw_weights = cached
    if total_trees == 0:
        return {}   # W disconnected (i, j not grid-adjacent) - this pair always self-loops

    split_prob = {}
    for side_nodes, w in raw_weights.items():
        new_a = apply_split(a, W, i, j, side_nodes)
        s = cut_edges(new_a, U_sub, V_sub)   # grid edges between the two proposed districts
        B = canonicalize(new_a)
        split_prob[B] = split_prob.get(B, 0.0) + (w / total_trees) / s
    return split_prob


def grid_symmetries(rows, cols):
    def idx(i, j): return i * cols + j
    def make_perm(transform):
        perm = np.empty(rows * cols, dtype=int)
        for i in range(rows):
            for j in range(cols):
                ni, nj = transform(i, j)
                perm[idx(ni, nj)] = idx(i, j)
        return perm
    transforms = {
        'identity': lambda i, j: (i, j),
        'flip_h':   lambda i, j: (i, cols - 1 - j),
        'flip_v':   lambda i, j: (rows - 1 - i, j),
        'rot180':   lambda i, j: (rows - 1 - i, cols - 1 - j),
    }
    if rows == cols:
        transforms.update({
            'rot90':          lambda i, j: (j, rows - 1 - i),
            'rot270':         lambda i, j: (cols - 1 - j, i),
            'transpose':      lambda i, j: (j, i),
            'anti_transpose': lambda i, j: (cols - 1 - j, rows - 1 - i),
        })
    return {name: make_perm(t) for name, t in transforms.items()}

def rotation_reflection_groups(assignments, plan_index, m, n):
    N = len(assignments)
    parent = list(range(N))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    perms = grid_symmetries(m, n)
    for idx, line in enumerate(assignments):
        a = parse(line)
        for perm in perms.values():
            key = canonicalize(a[perm])
            if key in plan_index:
                r1, r2 = find(idx), find(plan_index[key])
                if r1 != r2: parent[r1] = r2
    groups = {}
    for idx in range(N):
        groups.setdefault(find(idx), []).append(idx)
    return list(groups.values())

def dual_graph_groups(assignments, U, V):
    graphs = []
    for line in assignments:
        a = parse(line)
        G = nx.Graph()
        G.add_nodes_from(np.unique(a))
        G.add_edges_from(district_adjacency(a, U, V))
        graphs.append(G)
    buckets = {}
    for idx, G in enumerate(graphs):
        h = nx.weisfeiler_lehman_graph_hash(G)
        buckets.setdefault(h, []).append(idx)
    groups = []
    for members in buckets.values():
        remaining = list(members)
        while remaining:
            rep = remaining.pop()
            cls, still = [rep], []
            for other in remaining:
                (cls if nx.is_isomorphic(graphs[rep], graphs[other]) else still).append(other)
            remaining = still
            groups.append(cls)
    return groups

def lump_matrix(P, groups):
    G = len(groups)
    P_lumped = np.zeros((G, G))
    for I, src in enumerate(groups):
        for J, dst in enumerate(groups):
            P_lumped[I, J] = np.mean([P[a, dst].sum() for a in src])
    return P_lumped

def group_representatives(groups, assignments, m, n):
    return [parse(assignments[g[0]]).reshape(m, n) for g in groups]

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n
        self.max_size = 1
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            if self.size[rx] < self.size[ry]:
                rx, ry = ry, rx
            self.parent[ry] = rx
            self.size[rx] += self.size[ry]
            self.components -= 1
            self.max_size = max(self.max_size, self.size[rx])

def trace_components_adding(N, edge_order):
    uf = UnionFind(N)
    counts = []
    for u, v in edge_order:
        uf.union(u, v)
        counts.append(uf.components)
    return counts


def trace_largest_component_removing(N, removal_order):
    uf = UnionFind(N)
    trace = []
    for u, v in reversed(removal_order):   # process removal in reverse, as additions
        uf.union(u, v)
        trace.append(uf.max_size)
    trace.reverse()
    return trace   # Fig. 7b-style: trace[i] = largest component after removing removal_order[:i]


In [ ]:
filename = 'enumerated_grids/enum_Any[4,4]_4_rc.txt'
# filename = 'enumerated_grids/enum_Any[6,6]_6_rc.txt'
# filename = 'enumerated_grids/enum_Any[5,5]_5_rc.txt'

In [ ]:
assignments = load_grid(filename)
m, n = map(int, re.search(r'\[(\d+),(\d+)\]', filename).groups())
plan_index = build_plan_index(assignments,m,n)

In [ ]:
N = len(assignments)
P = np.zeros((N, N))

U, V = build_edges(m, n)
k = len(np.unique(parse(assignments[0])))
target_size = (m * n) // k
D_total = k * (k - 1) // 2
shape_cache = {}

for idx, line in enumerate(assignments):
    a = parse(line)
    diag = 0.0
    for i, j in combinations(np.unique(a), 2):
        Wi = district_nodes(a, i)
        Wj = district_nodes(a, j)
        W = np.union1d(Wi, Wj)
        U_sub, V_sub = induced_edges(U, V, W)

        dist = split_distribution(a, W, U_sub, V_sub, i, j, target_size, shape_cache)
        diag += (1.0 - sum(dist.values())) / D_total
        for B, p in dist.items():
            P[idx, plan_index[B]] += p / D_total
    P[idx, idx] += diag


In [ ]:
D = shortest_path(csr_matrix(P), method='D', directed=True)

edges = [(x, y) for x, y in zip(*np.nonzero(P)) if x != y]
curvatures = {(x, y): ollivier_ricci_edge(P, D, x, y) for x, y in edges}

spectral_gap(P)
    

In [ ]:
undirected = {}
for (x, y), kappa in curvatures.items():
    key = (min(x, y), max(x, y))
    undirected.setdefault(key, []).append(kappa)
undirected_curv = {e: np.mean(vs) for e, vs in undirected.items()}

edges_increasing = sorted(undirected_curv, key=lambda e: undirected_curv[e])   # most negative first
edges_decreasing = edges_increasing[::-1]
edges_random = random.sample(edges_increasing, len(edges_increasing))


In [ ]:
G = nx.DiGraph()
for x, y in zip(*np.nonzero(P)):
    if x != y:
        G.add_edge(x, y, weight=P[x, y])

edge_betweenness = nx.edge_betweenness_centrality(G, weight='weight', normalized=False)

In [ ]:
trans_probs = []
for key in curvatures:
    trans_probs.append(P[key[0]][key[1]])
len(trans_probs)

In [ ]:
connect_dict = {}
curvature_dict = {}

curvature_dict['curvature'] = list(curvatures.values())
curvature_dict['betweeness'] = list(edge_betweenness.values())
curvature_dict['transition prob'] = trans_probs
curvature_df_4 = pd.DataFrame(curvature_dict)

connect_dict['increasing_add'] = trace_components_adding(N, edges_increasing)
connect_dict['decreasing_add'] = trace_components_adding(N,edges_decreasing)
connect_dict['increasing_remove'] = trace_largest_component_removing(N, edges_increasing)
connect_dict['edge_num'] = np.arange(1,len(edges_increasing)+1,1)
connect_dict['random_remove'] = trace_largest_component_removing(N,edges_random)
connect_df_4 = pd.DataFrame(connect_dict)

In [ ]:
curve_betweeness = alt.Chart(curvature_df_4).mark_circle(color='green').encode(
    x = alt.X('betweeness:Q',title='Betweeness Centrality'),
    y = alt.Y('curvature:Q',title='Curvature')
)

curve_transition = alt.Chart(curvature_df_4).mark_circle(color='orange').encode(
    x = alt.X('transition prob:Q',title='Transition probability'),
    y = alt.Y('curvature:Q',title='Curvature')
)

curve_betweeness | curve_transition

In [ ]:
import altair as alt


increasing_add = alt.Chart(connect_df_4).mark_point(opacity=0.5).encode(
    x=alt.X('edge_num:Q', title='# edges added'),
    y=alt.Y('increasing_add:Q', title='# connected components'),
    color=alt.Color('label:N').scale(domain=['Increasing', 'Decreasing'], range=['blue', 'pink']).title('Strategy')
).transform_calculate(label="'Increasing'").properties(title='Largest components under edge addition')

decreasing_add = alt.Chart(connect_df_4).mark_point(opacity=0.5).encode(
    x=alt.X('edge_num:Q'),
    y=alt.Y('decreasing_add:Q'),
    color=alt.Color('label:N')
).transform_calculate(label="'Decreasing'")


increasing_remove = alt.Chart(connect_df_4).mark_point(opacity=0.5).encode(
    x=alt.X('edge_num:Q', title='# edges removed'),
    y=alt.Y('increasing_remove:Q', title='Largest connected component'),
    color=alt.Color('label:N').scale(domain=['Increasing', 'Random'], range=['blue', 'red']).title('Strategy')
).transform_calculate(label="'Increasing'")

random_remove = alt.Chart(connect_df_4).mark_point(opacity=0.4).encode(
    x=alt.X('edge_num:Q'),
    y=alt.Y('random_remove:Q'),
    color=alt.Color('label:N')
).transform_calculate(label="'Random'").properties(title='Number of components under edge removal')


add = increasing_add + decreasing_add
remove = increasing_remove + random_remove


(add | remove).resolve_scale(color='independent')
add.save('4x4addedge.png',ppi=300)
remove.save('4x4removeedge.png',ppi=300)


In [ ]:
orbits = rotation_reflection_groups(assignments, plan_index, m, n)
P_orbits = lump_matrix(P, orbits)
orbit_reps = group_representatives(orbits, assignments, m, n)

In [ ]:
D = shortest_path(csr_matrix(P_orbits), method='D', directed=True)
print(spectral_gap(P_orbits))
e = spectra(P_orbits)
edges = list(zip(*np.nonzero(P_orbits)))
curvatures = {(x, y): ollivier_ricci_edge((P_orbits), D, x, y) for x, y in edges}
curvatures


In [ ]:
sign = pd.cut(
    curvature_df_4['curvature'],
    bins=[-np.inf, -1e-9, 1e-9, np.inf],
    labels=['negative', 'zero', 'positive'],
)
sign_counts = sign.value_counts().reindex(['negative', 'zero', 'positive']).reset_index()
sign_counts.columns = ['sign', 'count']

sign_chart = alt.Chart(sign_counts).mark_bar().encode(
    x=alt.X('sign:N', sort=['negative', 'zero', 'positive'], title=None),
    y=alt.Y('count:Q', title='# of edges'),
    color=alt.Color(
        'sign:N',
        sort=['negative', 'zero', 'positive'],
        scale=alt.Scale(domain=['negative', 'zero', 'positive'], range=['#3b82f6', '#9ca3af', '#ef4444']),
        legend=None,
    ),
    tooltip=['sign:N', 'count:Q'],
).properties(width=200, height=250, title='Sign of curvature on 6x6 grid')

sign_chart.save('6x6curvaturesign.png',ppi=300)


In [ ]:
hist_chart = alt.Chart(curvature_df_4).mark_bar().encode(
    x=alt.X('curvature:Q', bin=alt.Bin(step=0.1), title='Ricci curvature'),
    y=alt.Y('count():Q', title='# of edges'),
    tooltip=[alt.Tooltip('count():Q', title='# of edges')],
).properties(width=500, height=250, title=f'{"4x4"} grid curvature distribution ')
hist_chart.save('4x4curvehist.png',ppi=300)